# Suite ANY — Values of any type

`Schemas.OfAny.Value` is the kind of a value of any type: built with `types`, it is restricted to them, and with none,
it is unrestricted. A restricted value is written bare, its type the one whose
plain form it takes; an unrestricted one is tagged with its type, a scalar's token name or a registered schema's name,
and is assigned as a value, built standalone through the store where it is not a scalar.

In [ ]:
import { Builders as B, Comparison as C, JSON, Plain, Proxies, Schemas as S, Validators, YAML } from "@mbse/schemas/Framework";
import { DecodeError, ReadOnlyError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, map, raises, root_of, same } from "./support.js";

const store: any = new B.Store();
const Int = store.Schemas.OfInt.as_schema(); // the store's own
const Text = store.Schemas.OfStr.as_schema(); // the store's own
const Float = store.Schemas.OfFloat.as_schema(); // the store's own
const Bytes = store.Schemas.OfBytes.as_schema(); // the store's own
const Bool = store.Schemas.OfBool.as_schema(); // the store's own
const Phone = store.Schemas.OfObject((o: any) => o.name("Phone").properties([(t: any) => t.name("number").type([Text])]));
const Code = store.Schemas.OfUnion((u: any) => (
  u
    .name("Code")
    .branches([
      (t: any) => t.name("n").type([Int]),
      (t: any) => t.name("s").type([Text]),
    ])
));
const Stamp = store.Schemas.OfIntersection((i: any) => (
  i
    .name("Stamp")
    .parts([
      (t: any) => t.name("at").type([Int]),
      (t: any) => t.name("by").type([Text]),
    ])
));
const Tags = store.Schemas.OfContainer((c: any) => c.name("Tags").index([Int]).element([Text]));
const Labels = store.Schemas.OfContainer((c: any) => c.name("Labels").index([Text]).element([Text])); // a dict by text: a mapping
const Cell = store.Schemas.OfObject((o: any) => o.properties([
  (t: any) => t.name("r").type([Int]),
  (t: any) => t.name("c").type([Int]),
]));
const Grid = store.Schemas.OfContainer((c: any) => c.index([Cell]).element([Int])); // a tensor: nested arrays, or flat with its shape
const One = store.Schemas.OfAny((a: any) => a.types([Int, Text, Phone]));
const Free = store.Schemas.OfAny((a: any) => a);
const Card = store.Schemas.OfObject((o: any) => (
  o
    .name("Card")
    .ref(true)
    .pattern(String.raw`card-(\d+)`)
    .properties([
      (t: any) => t.name("one").type([One]),
      (t: any) => t.name("free").type([Free]),
      (t: any) => t.name("many").type([store.Schemas.OfContainer((c: any) => c.index([Int]).element([Free]))]),
      (t: any) => t.name("tags").type([store.Schemas.OfAny((a: any) => a.types([Bool, Tags]))]),
    ])
));

/** What a property holds, read through its visitor: a scalar, or a visitor's schema's name; null where absent. */
function held(prop: any): unknown {
  const found: unknown[] = [];
  const named = (v: any) => {
    if (v.as_schema() !== null) found.push(v.as_schema().name);
    return v;
  };
  prop.as_value((a: any) => a.as_scalar((s: any) => {
    if (s.get() !== null) found.push(s.get());
    return s;
  }).as_object(named).as_union(named).as_intersection(named).as_container(named));
  return found.length > 0 ? found[0] : null;
}

/** A schema's plain forms, sorted. */
const forms = (schema: unknown) => [...S.plain_forms(schema)].sort();

## ANY-01 · An `OfAny` restricted to the types selected, or to none; its types told apart on the wire

In [ ]:
{
  assert(One instanceof S.OfAny.Value && same(One.types, [Int, Text, Phone]) && One.validate().length === 0);
  assert(Free.types.length === 0 && Free.validate().length === 0 && Card.validate().length === 0);
  const only = store.Schemas.OfAny((a: any) => a.types([Int])); // one type is an OfAny still: where one type is meant, give it directly
  assert(only instanceof S.OfAny.Value && same(only.types, [Int]) && store.Schemas.OfProperty((p: any) => p.type([Int])).type === Int);
  assert(equal([forms(Float), forms(Bytes), forms(Labels), forms(Tags), forms(Grid)],
    [["float", "text"], ["text"], ["mapping"], ["list"], ["list", "mapping"]])); // NaN is text, bytes base64, a tensor flat
  assert(same(forms(One), ["int", "mapping", "text"]) && same(forms(Free), ["mapping"])); // unrestricted: tagged
  assert(same(forms(new S.OfScalar.Data(new S.OfScalar.Token("ccpp", "x"))), ["bool", "float", "int", "list", "mapping", "text"]));
  const loop = store.Schemas.OfApply((a: any) => a);
  store.Schemas.OfApply(loop, (a: any) => a.of([loop]));
  assert(forms(loop).length === 0); // an application that applies itself has no values
  assert(same(new S.OfAny.Value([loop, new S.OfScalar.Data("weird")]).validate(), [
    "a schema: an application cannot apply itself", "a schema: unsupported scalar type 'weird'"]));
  assert(same(store.Schemas.OfAny((a: any) => a.types([Tags, Grid])).validate(), [
    "an OfAny's types Tags and a container are not told apart on the wire"]));
  assert(same(store.Schemas.OfAny((a: any) => a.types([Cell, Labels])).validate(), [
    "an OfAny's types a value object and Labels are not told apart on the wire"]));
  assert(One.equals(One) && !One.equals(Free)); // schemas by identity
  const Either = store.Schemas.OfUnion((u: any) => u.branches([
    (t: any) => t.name("a").type([Int]),
    (t: any) => t.name("b").type([Text]),
  ]));
  const Both = store.Schemas.OfIntersection((i: any) => i.parts([
    (t: any) => t.name("a").type([Int]),
    (t: any) => t.name("b").type([Text]),
  ]));
  assert(same(store.Schemas.OfAny((a: any) => a.types([Float, Text, Phone, Code, Labels, Either, Both, Tags])).validate(), [
    "an OfAny's types Schemas.OfFloat and Schemas.OfStr are not told apart on the wire",
    "an OfAny's types Phone and Code are not told apart on the wire",
    "an OfAny's types Phone and Labels are not told apart on the wire",
    "an OfAny's types Phone and a union are not told apart on the wire",
    "an OfAny's types Phone and an intersection are not told apart on the wire",
    "an OfAny's types Code and Labels are not told apart on the wire",
    "an OfAny's types Code and a union are not told apart on the wire",
    "an OfAny's types Code and an intersection are not told apart on the wire",
    "an OfAny's types Labels and a union are not told apart on the wire",
    "an OfAny's types Labels and an intersection are not told apart on the wire",
    "an OfAny's types a union and an intersection are not told apart on the wire"]));
  const unnamed = store.Schemas.OfScalar((s: any) => s.token("float")); // an unnamed scalar is shown by its token's name
  assert(same(store.Schemas.OfAny((a: any) => a.types([unnamed, Text])).validate(), [
    "an OfAny's types float and Schemas.OfStr are not told apart on the wire"]));
  raises(TypeError, () => store.Schemas.OfAny((a: any) => a.types([3n])), "link 'schema' cannot link 3"); // the builders link schemas, nothing else
  assert(same(new S.OfAny.Value([3n, One, new S.OfScalar.Data(BigInt, { name: "1x" }), Grid]).validate(), [ // data they refuse to build
    "an OfAny's types are schemas, got 3", "an OfAny's types are schemas of other kinds, not an OfAny",
    "1x: name '1x' is not identifiers separated by dots, e.g. 'crm.Contact'"]));
  const Thing = store.Schemas.OfObject((o: any) => o.name("Thing").ref(true));
  assert(same(store.Schemas.OfObject((o: any) => o.properties([
    (t: any) => t.name("x").type([store.Schemas.OfAny((a: any) => a.types([Int, Thing]))]),
  ])).validate(), ["property 'x': a reference object schema cannot be a property's type"])); // a value only
}

## ANY-02 · Proxies: a restricted value written through the visitors as its type is, inferred; an unrestricted one given as a value, built standalone where it is not a scalar

In [ ]:
let card = store.Card((c: any) => c.one(3n).free("x"));
assert(held(card.one) === 3n && held(card.free) === "x" && card.one.get() === 3n);
card = store.Card((c: any) => (
  c
    .one((p: any) => p.as_object((o: any) => o.number("1"))) // the one type of its kind
    .tags((p: any) => p.as_value((a: any) => a.as_container((t: any) => t.at(0n).as_scalar("a"))))
));
assert(held(card.one) === "Phone" && held(card.tags) === "Tags" && card.one.get() === null);
raises(TypeError, () => store.Card((c: any) => c.one(store.Tags((t: any) => t))), "property 'one' does not hold a value of Tags");
raises(TypeError, () => Proxies.OfObject.Builder(store, Phone, "Phone"),
  "'Phone' is a value object schema; build a value through its owner, or standalone through the store");
raises(ReadOnlyError, () => store.Card((c: any) => c.one((p: any) => p.as_value((a: any) => a.as_union((u: any) => u.n(1n))))),
  "property 'one' does not hold a union value");
// Unrestricted: a scalar, or a value, standalone or another's; a Spec has no schema to build by.
const phone = store.Phone((p: any) => p.number("2"));
const tags = store.Tags((t: any) => t.at(0n).as_scalar("t"));
assert(Proxies.owner_of(phone) === null && tags.as_schema() === Tags);
const values = [phone, store.Code((c: any) => c.n(1n)), store.Stamp((s: any) => s.at(1n).by("me")), tags, card.one];
assert(same(values.map((value) => held(store.Card((c: any) => c.free(value)).free)), ["Phone", "Code", "Stamp", "Tags", "Phone"]));
assert(held(store.Card((c: any) => c.free((p: any) => p.as_value((a: any) => a.as_object(card.one)))).free) === "Phone");
{
  const fresh = store.Phone((p: any) => p.number("3"));
  const [first, second] = [store.Card((c: any) => c.free(fresh)), store.Card((c: any) => c.free(fresh))];
  const phones: any[] = [];
  for (const each of [first, second]) each.free.as_value((a: any) => a.as_object((o: any) => phones.push(o) && o));
  assert(phones[0] === fresh && phones[1] !== fresh && Proxies.owner_of(fresh) === first); // adopted once, then copied
  const unbuilt = "property 'free' holds a value of any type: give a value, which carries its schema, not a Spec";
  raises(ReadOnlyError, () => store.Card((c: any) => c.free((p: any) => p.as_value((a: any) => a.as_object((o: any) => o.number("1")))))
    , unbuilt);
  raises(ReadOnlyError, () => store.Card((c: any) => c.free((p: any) => p.as_value((a: any) => a.as_container((c: any) => c.remove(0n)))))
    , unbuilt);
  const read: unknown[] = []; // a Spec of another kind than the value held reads it as empty, and writes nothing
  store.Card((c: any) => c.one(3n).one((p: any) => p.as_value((a: any) => a.as_object((o: any) => read.push(o.as_schema()) && o))));
  assert(same(read, [null]));
  const found: unknown[] = [];
  card.free.as_value((a: any) => a.as_scalar((s: any) => found.push([s.as_schema(), s.get()]) && s));
  assert(equal(found, [[Free, null]])); // nothing held: a scalar of any type, absent
  const listed = store.Card((c: any) => c.many((p: any) => p.as_value((a: any) => a.as_container(
    (c: any) => c.at(0n).as_scalar(1n) && c.at(1n).as_object(phone)))));
  const elements: [unknown, unknown][] = [];
  listed.many.as_value((a: any) => a.as_container((c: any) => c.as_entries((it: Iterable<[unknown, unknown]>) => elements.push(...it))));
  assert(same(elements.map(([index]) => index), [0n, 1n]));
  raises(TypeError, () => store.Card((c: any) => c.one(listed.many)), "property 'one' does not hold a value of a container");
}

## ANY-03 · Snapshots: a restricted value bare, an unrestricted one tagged, read back; malformed ones refused, with a path

In [ ]:
const other: any = new B.Store(); // another store of the same schemas
for (const schema of [Phone, Code, Stamp, Tags, Labels, Card]) other.register(schema);
const many = (c: any) => {
  c.at(0n).as_scalar(2.5);
  c.at(1n).as_object(phone);
  c.at(3n).as_container(tags); // nothing at 2
  return c;
};
card = store.Card((c: any) => (
  c
    .one((p: any) => p.as_object((o: any) => o.number("1")))
    .free(store.Code((x: any) => x.s("c")))
    .many((p: any) => p.as_value((a: any) => a.as_container(many)))
    .tags(true)
));
assert(equal(root_of(Plain.ToPlain(store).OfObject(Card, card)), map({ one: { number: "1" }, free: { Code: { s: "c" } },
  tags: true, many: [{ float: 2.5 }, { Phone: { number: "2" } }, null, { Tags: ["t"] }] })));
for (const [encode, decode] of [[JSON.ToJSON, JSON.FromJSON], [YAML.ToYAML, YAML.FromYAML]] as const) {
  const written = (encode as any)(store).OfObject(Card, card);
  assert((encode as any)(other).OfObject(Card, (decode as any)(other).OfObject(Card, written)) === written);
}
assert(equal(root_of(Plain.ToPlain(store).OfObject(Card, store.Card((c: any) => c.one("s").free(new Uint8Array([1]))))),
  map({ one: "s", free: { bytes: "AQ==" } })));
assert(Plain.FromPlain(store).OfObject(Card, Plain.ToPlain(store).OfObject(Card, card)) === card); // reloading: an update
{
  const back: any = Plain.FromPlain(other).OfObject(Card, map({ root: "Card/card-8", objects: { "Card/card-8": { one: 3n, tags: ["a"] } } }));
  assert(held(back.one) === 3n && held(back.tags) === "Tags"); // each read as the one type its plain form fits
  const text_: any = Plain.FromPlain(other).OfObject(Card, map({ root: "Card/card-7", objects: { "Card/card-7": { one: "s" } } }));
  assert(held(text_.one) === "s");
}
const refused = (values: Record<string, unknown>, message: string) => raises(DecodeError,
  () => Plain.FromPlain(other).OfObject(Card, map({ root: "Card/card-9", objects: { "Card/card-9": values } })),
  `$.objects["Card/card-9"]${message}`);
refused({ one: true }, ".one: a value of any type takes the plain form of one of its types, got bool");
refused({ one: 2.5 }, ".one: a value of any type takes the plain form of one of its types, got float");
refused({ one: null }, ".one: a value of any type takes the plain form of one of its types, got NoneType");
refused({ free: 3n }, ".free: a value of any type is {type: value}, its type a scalar's token name or a registered schema's name");
refused({ free: { Nope: 1n } }, ".free.Nope: no schema registered as 'Nope'");
refused({ free: { Card: {} } }, ".free.Card: 'Card' is not a value's schema");
refused({ free: { int: "x" } }, ".free.int: expected int, got str");
raises(TypeError, () => Plain.ToPlain(store).OfObject(Card, store.Card((c: any) => c.one(2.5))),
  "a value of any type holds one of its types; float is not one"); // builders do not check scalars' types
const Point = store.Schemas.OfObject((o: any) => o.properties([(t: any) => t.name("x").type([Int])])); // unnamed: not registered
const Spot = store.Schemas.OfObject((o: any) => o.name("Spot").ref(true).properties([(t: any) => t.name("at").type([Point])]));
const loose = store.Card((c: any) => c.free(store.Spot((s: any) => s.at((p: any) => p.as_object((o: any) => o.x(1n)))).at));
raises(TypeError, () => Plain.ToPlain(store).OfObject(Card, loose), "a value of any type is of a registered schema; None is not registered");

## ANY-04 · Validation and comparison under the type a value holds; uniqueness by value

In [ ]:
{
  const validate = Validators.Validate(store);
  assert(validate(Card, card).length === 0);
  assert(same(validate(Card, store.Card((c: any) => c.one(2.5).many((p: any) => p.as_value((a: any) => a.as_container(
    (c: any) => c.at(0n).as_object(phone)))))), ["Card#0.one: float is not one of its types"]));
  assert(same(validate(Card, loose), ["Card#0.free: a value of any type is of a registered schema; None is not registered"]));
  assert(same(validate(Card, store.Card((c: any) => c.free(store.Code((x: any) => x.n("x"))))), ["Card#0.free.n: expected int, got str"]));
  const compare = (a: any, b: any) => new C.OfObject(Card, a).compare(new C.OfObject(Card, b));
  assert(compare(store.Card((c: any) => c.one(3n).free(phone)),
    store.Card((c: any) => c.one(3n).free(store.Phone((p: any) => p.number("2"))))) === 0);
  assert(compare(store.Card((c: any) => c.one(3n)), store.Card((c: any) => c.one("3"))) === null); // of different types: incomparable
  assert(compare(store.Card((c: any) => c.free(phone)), store.Card((c: any) => c.free(store.Code((x: any) => x.s("2"))))) === null);
  const Holds = store.Schemas.OfRelation((r: any) => (
    r
      .name("Holds")
      .links([
        (t: any) => t.name("card"),
        (t: any) => t.name("other"),
      ])
      .properties([
        (t: any) => t.name("what").type([Free]),
        (t: any) => t.name("note").type([Text]),
      ])
      .uniques("note")
  ));
  const Holder = store.Schemas.OfObject((o: any) => (
    o
      .name("Holder")
      .ref(true)
      .adjacencies([
        (t: any) => t.name("holds").relation([Holds]).me("card"),
        (t: any) => t.name("held").relation([Holds]).me("other"),
      ])
  ));
  const target = store.Holder((h: any) => h);
  const holder = store.Holder((h: any) => h.holds((a: any) => (
    a
      .insert((e: any) => e.other(target).what(phone).note("a"))
      .insert((e: any) => e.other(target).what(store.Phone((p: any) => p.number("2"))).note("b"))
      .insert((e: any) => e.other(target).what(1n).note("c"))
      .insert((e: any) => e.other(target).note("d"))
  )));
  assert(same(validate(Holder, holder), [
    "unique(note) violated: entries agreeing on ['card', 'other', 'what'] differ on ['note']"])); // equal values: one key
  for (const schema of [Holds, Holder]) other.register(schema);
  const snapshot = Plain.ToPlain(store).Reachable(Holder, holder);
  assert(equal((root_of(snapshot) as any).get("holds").map((e: any) => e.get("what") ?? null),
    [map({ Phone: { number: "2" } }), map({ Phone: { number: "2" } }), map({ int: 1n }), null]));
  assert(JSON.dumps(Plain.ToPlain(other).Reachable(Holder, Plain.FromPlain(other).Reachable(Holder, snapshot) as any))
    === JSON.dumps(snapshot));
  assert(Plain.FromPlain(store).Reachable(Holder, snapshot) === holder && holder.holds.length === 4); // reloading adds none
  assert(new C.OfObject(Card, card).compare(new C.OfObject(Card, Plain.FromPlain(other).OfObject(Card,
    Plain.ToPlain(store).OfObject(Card, card)) as any)) === 0); // a container of values of any type, element by element
}

## ANY-05 · Snapshots write a value of any type's types, linked, and read them back

In [ ]:
{
  const typed: any = new B.Store(); // a snapshot holds every schema of its store: these, typed by the basic scalars
  const Phoned = typed.Schemas.OfObject((o: any) => o.name("Phone").properties([(t: any) => t.name("number").type([typed.Schemas.OfStr.as_schema()])]));
  typed.Schemas.OfObject((o: any) => (
    o
      .name("Card")
      .ref(true)
      .properties([
        (t: any) => t.name("one").type([typed.Schemas.OfAny((a: any) => a.types([typed.Schemas.OfInt.as_schema(), typed.Schemas.OfStr.as_schema(), Phoned]))]),
        (t: any) => t.name("free").type([typed.Schemas.OfAny((a: any) => a)]),
      ])
  ));
  const log = Plain.ToPlain(typed).Snapshot();
  const back: any = new B.Store();
  const made = Plain.FromPlain(back).Snapshot(log);
  assert(JSON.dumps(Plain.ToPlain(back).Snapshot()) === JSON.dumps(log)); // read back, written again: the same
  const [one, free] = ["one", "free"].map((name) => back.registered("Card").properties.get(name).type);
  assert(one instanceof S.OfAny.Value && one.types[2] === back.registered("Phone") && one.types[0] === back.Schemas.OfInt.as_schema());
  assert(free.types.length === 0);
}